# Base Paper Replication — Federated Learning
**Reference:** Mohan Raj, Morley, Eslami (2024). *Federated Learning for Diabetic Retinopathy Diagnosis.* arXiv:2411.00869

**Setup sesuai base paper:**
- Model: EfficientNetB0 (ImageNet pretrained)
- Optimizer: SGD + momentum (lr=0.001)
- Clients: DDR (H1) + EyePACS (H2) sebagai FL training institutions
- Unseen: APTOS (H3) sebagai simulated under-resourced institution
- APTOS degraded: JPEG compression quality 30-50
- Split: 90:10 per training client
- FL: FedAvg dengan dataset-size-weighted aggregation
- No class weights

> **Catatan:** Ini replication/comparison notebook.
> Paper lo sendiri pakai setup yang berbeda dan lebih rigorous.

## 1. GPU Check

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Imports

In [ ]:
import os, json, random, time, gc, warnings, io
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.notebook import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score, roc_curve, auc,
    cohen_kappa_score
)
from sklearn.preprocessing import label_binarize

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import EfficientNetB0

print("TF:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

## 4. Configuration

In [ ]:
# ── REPRODUCIBILITY ──────────────────────────────────────────────
SEED             = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# ── MODEL ─────────────────────────────────────────────────────────
MODEL_NAME       = "BasePaper_EfficientNetB0"
EXPERIMENT_NAME  = "fl_basepaper"

# ── TRAINING (sesuai base paper) ──────────────────────────────────
IMG_SIZE         = (224, 224)
NUM_CLASSES      = 5
BATCH_SIZE       = 32
LR               = 0.001
MOMENTUM         = 0.9
LOCAL_EPOCHS     = 10            # base paper: local epochs per round
FL_ROUNDS        = 10            # base paper: communication rounds
STAGE1_ROUNDS    = 5             # rounds 1-5: frozen
STAGE2_ROUNDS    = 5             # rounds 6-10: unfrozen
UNFREEZE_N       = 10
DROPOUT          = 0.4
L2               = 1e-4

# ── SPLIT ─────────────────────────────────────────────────────────
TRAIN_RATIO      = 0.90
TEST_RATIO       = 0.10

# ── NOISE: JPEG (sesuai base paper) ───────────────────────────────
JPEG_QUALITY_MIN = 30
JPEG_QUALITY_MAX = 50

# ── ROLES ─────────────────────────────────────────────────────────
TRAIN_CLIENTS    = ["DDR", "EyePACS"]  # FL training institutions
UNSEEN_CLIENT    = "APTOS"             # simulated under-resourced

CLASS_NAMES      = ["No DR","Mild","Moderate","Severe","Proliferative"]

# ── PATHS ─────────────────────────────────────────────────────────
PROJECT  = "/content/drive/MyDrive/Binus/Semester_4/Research_Methodology"
DS_BASE  = "/content/drive/MyDrive/S-Class/Orion/OrionFL"

APTOS_NPZ = f"{DS_BASE}/APTOS_2019/preprocessed/orion_dr_224.npz"
DDR_NPZ   = f"{DS_BASE}/DDR_Dataset/DDR_dataset_224.npz"
EYE_NPZ   = f"{DS_BASE}/EyePACS_Dataset/training/EyePACS_dataset_224.npz"
EYE_CSV   = f"{DS_BASE}/EyePACS_Dataset/training/trainLabels.csv"

RES      = f"{PROJECT}/Result/{MODEL_NAME}/{EXPERIMENT_NAME}"
MDL_DIR  = f"{RES}/models"
LOG_DIR  = f"{RES}/logs"
FIG_DIR  = f"{RES}/figures"
for d in [MDL_DIR, LOG_DIR, FIG_DIR]: os.makedirs(d, exist_ok=True)

print(f"Model       : {MODEL_NAME}")
print(f"Optimizer   : SGD lr={LR}, momentum={MOMENTUM}")
print(f"FL          : {FL_ROUNDS} rounds × {LOCAL_EPOCHS} local epochs")
print(f"Train on    : {TRAIN_CLIENTS}")
print(f"Unseen test : {UNSEEN_CLIENT}")
print(f"Results     : {RES}")

## 5. Load Datasets

In [ ]:
def to01(X):
    X = X.astype(np.float32)
    return X / 255.0 if X.max() > 1.0 else X

print("Loading DDR...")
d = np.load(DDR_NPZ, allow_pickle=True)
X_ddr, y_ddr = to01(d["X"]), d["y"].astype(np.int64)
print(f"  {X_ddr.shape}")

print("Loading EyePACS...")
d = np.load(EYE_NPZ, allow_pickle=True)
X_eye = to01(d["images"])
y_eye = pd.read_csv(EYE_CSV)["level"].values.astype(np.int64)
assert len(X_eye) == len(y_eye)
print(f"  {X_eye.shape}")

print("Loading APTOS (unseen)...")
d = np.load(APTOS_NPZ, allow_pickle=True)
X_apt, y_apt = to01(d["images"]), d["labels"].astype(np.int64)
print(f"  {X_apt.shape}")

## 6. Split 90:10 per Training Client

In [ ]:
def split_90_10(X, y, seed=SEED):
    return train_test_split(X, y, test_size=TEST_RATIO,
                            random_state=seed, stratify=y)

Xtr_d, Xte_d, ytr_d, yte_d = split_90_10(X_ddr, y_ddr)
Xtr_e, Xte_e, ytr_e, yte_e = split_90_10(X_eye, y_eye)

# Client data (FL)
client_X   = [Xtr_d, Xtr_e]
client_y   = [ytr_d, ytr_e]
client_sz  = [len(ytr_d), len(ytr_e)]

# In-distribution test
X_test_ind = np.concatenate([Xte_d, Xte_e])
y_test_ind = np.concatenate([yte_d, yte_e])
src_ind    = np.array(["DDR"]*len(yte_d) + ["EyePACS"]*len(yte_e))

# Unseen (APTOS)
X_test_uns = X_apt
y_test_uns = y_apt

print(f"DDR     train:{len(ytr_d):>6,}  test:{len(yte_d):>5,}")
print(f"EyePACS train:{len(ytr_e):>6,}  test:{len(yte_e):>5,}")
print(f"APTOS   unseen: {len(y_apt):,}")
print(f"FedAvg weights: DDR={client_sz[0]/sum(client_sz):.3f}  EyePACS={client_sz[1]/sum(client_sz):.3f}")

del X_ddr, X_eye, X_apt; gc.collect()

## 7. JPEG Degradation (APTOS)

In [ ]:
def jpeg_compress(img_float, quality):
    img_uint8 = (img_float * 255).clip(0,255).astype(np.uint8)
    buf = io.BytesIO()
    Image.fromarray(img_uint8).save(buf, format="JPEG", quality=quality)
    buf.seek(0)
    return np.array(Image.open(buf)).astype(np.float32) / 255.0

def make_jpeg_degraded(X, seed=SEED):
    print(f"  Applying JPEG compression (q={JPEG_QUALITY_MIN}-{JPEG_QUALITY_MAX})...")
    rng = np.random.default_rng(seed); out = np.empty_like(X)
    for i in range(len(X)):
        out[i] = jpeg_compress(X[i], int(rng.integers(JPEG_QUALITY_MIN, JPEG_QUALITY_MAX+1)))
    print(f"  Done.")
    return out

print("Generating JPEG-degraded APTOS...")
X_test_uns_deg = make_jpeg_degraded(X_test_uns)

## 8. Model Builder + FedAvg + TF Dataset Helpers

In [ ]:
AUT = tf.data.AUTOTUNE

def make_ds(X, y, shuffle=False, seed=None):
    ds = tf.data.Dataset.from_tensor_slices((X,y))
    if shuffle: ds = ds.shuffle(len(X), seed=seed)
    return ds.batch(BATCH_SIZE).prefetch(AUT)

def build_model(stage=1):
    base = EfficientNetB0(include_top=False, weights="imagenet", input_shape=(224,224,3))
    base.trainable = False
    if stage == 2:
        base.trainable = True
        for l in base.layers[:-UNFREEZE_N]: l.trainable = False
        for l in base.layers[-UNFREEZE_N:]: l.trainable = True
        for l in base.layers:
            if isinstance(l, layers.BatchNormalization): l.trainable = False
    inp = keras.Input((224,224,3))
    x   = inp * 255.0   # [0,1] → [0,255] untuk EfficientNet internal rescaling
    x   = base(x, training=False)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.Dropout(DROPOUT)(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax",
                       kernel_regularizer=regularizers.l2(L2))(x)
    return keras.Model(inp, out)

def fedavg(weights_list, sizes):
    total = sum(sizes)
    return [np.sum([weights_list[i][li]*(sizes[i]/total) for i in range(len(sizes))],axis=0)
            for li in range(len(weights_list[0]))]

# Eval datasets
val_ds      = make_ds(X_test_ind, y_test_ind)       # in-distribution (used as val during FL)
test_ind_ds = make_ds(X_test_ind, y_test_ind)
test_uns_ds = make_ds(X_test_uns, y_test_uns)
test_deg_ds = make_ds(X_test_uns_deg, y_test_uns)

print("Model builder + FedAvg ready.")

## 9. Local Training Function

In [ ]:
def local_train(global_w, X, y, stage, seed):
    m = build_model(stage)
    m.set_weights(global_w)
    lr = LR if stage == 1 else LR/10
    m.compile(
        optimizer=keras.optimizers.SGD(learning_rate=lr, momentum=MOMENTUM),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"])
    ds = make_ds(X, y, shuffle=True, seed=seed)
    h  = m.fit(ds, epochs=LOCAL_EPOCHS, verbose=0)
    w  = m.get_weights()
    loss, acc = h.history["loss"][-1], h.history["accuracy"][-1]
    del m, ds; tf.keras.backend.clear_session(); gc.collect()
    return w, float(loss), float(acc)

print("local_train ready.")

## 10. FL Training Loop — Base Paper FedAvg

**Sesuai base paper:**
- Rounds 1-5: frozen backbone (Stage 1)
- Rounds 6-10: unfreeze top-10 (Stage 2)
- SGD optimizer
- No noise augmentation pada training clients
- APTOS hanya sebagai unseen evaluation

In [ ]:
# Init global model
gm = build_model(stage=1)
gm.compile(optimizer=keras.optimizers.SGD(LR, momentum=MOMENTUM),
            loss=keras.losses.SparseCategoricalCrossentropy(), metrics=["accuracy"])
gw = gm.get_weights(); del gm; tf.keras.backend.clear_session(); gc.collect()

fl_logs = []
print("="*65)
print(f"  Base Paper FedAvg | {FL_ROUNDS} rounds × {LOCAL_EPOCHS} local epochs")
print(f"  Clients: {TRAIN_CLIENTS} | SGD lr={LR}")
print(f"  Stage 1: rounds 1-{STAGE1_ROUNDS} | Stage 2: rounds {STAGE1_ROUNDS+1}-{FL_ROUNDS}")
print("="*65)

for rnd in tqdm(range(1, FL_ROUNDS+1), desc="FL Rounds"):
    t0    = time.time()
    stage = 1 if rnd <= STAGE1_ROUNDS else 2
    print(f"\n[Round {rnd:02d}/{FL_ROUNDS}] Stage {stage}")

    client_ws, client_logs = [], []
    for ci, (cname, cX, cy) in enumerate(zip(TRAIN_CLIENTS, client_X, client_y)):
        cw_, loss, acc = local_train(gw, cX, cy, stage, seed=SEED+rnd)
        client_ws.append(cw_)
        client_logs.append({"client":cname,"stage":stage,"loss":loss,"accuracy":acc,"n":int(client_sz[ci])})
        print(f"  ✓ {cname:<8} loss:{loss:.4f}  acc:{acc:.4f}  n:{client_sz[ci]:,}")

    gw = fedavg(client_ws, client_sz)

    # Eval global model
    gm = build_model(stage)
    gm.set_weights(gw)
    gm.compile(optimizer=keras.optimizers.SGD(LR if stage==1 else LR/10, momentum=MOMENTUM),
                loss=keras.losses.SparseCategoricalCrossentropy(), metrics=["accuracy"])
    vl, va = gm.evaluate(val_ds, verbose=0)
    elapsed = time.time()-t0
    print(f"  ▶ Val loss:{vl:.4f}  acc:{va:.4f}  ({elapsed:.1f}s)")

    fl_logs.append({"round":rnd,"stage":stage,"val_loss":float(vl),"val_accuracy":float(va),
                    "time_sec":round(elapsed,2),"clients":client_logs})
    gm.save(f"{MDL_DIR}/{EXPERIMENT_NAME}_round{rnd:02d}.keras")
    del client_ws; gc.collect()

print("\n"+"="*65+"\n  FL TRAINING COMPLETE\n"+"="*65)
gm.save(f"{MDL_DIR}/{EXPERIMENT_NAME}_final.keras")

pd.DataFrame([{"round":r["round"],"stage":r["stage"],"val_acc":r["val_accuracy"],
               "val_loss":r["val_loss"],"time_sec":r["time_sec"]} for r in fl_logs]
).to_csv(f"{LOG_DIR}/fl_round_summary.csv",index=False)

## 11. FL Training Curves

In [ ]:
rounds   = [r["round"] for r in fl_logs]
val_acc  = [r["val_accuracy"] for r in fl_logs]
val_loss = [r["val_loss"] for r in fl_logs]
sb       = next((r for r,s in zip(rounds,[r["stage"] for r in fl_logs]) if s==2),None)

fig,ax = plt.subplots(1,2,figsize=(13,5))
for a,data,title,clr in [(ax[0],val_acc,"Val Accuracy","#1565C0"),(ax[1],val_loss,"Val Loss","#1565C0")]:
    a.plot(rounds,data,marker="o",color=clr,lw=2)
    if sb: a.axvline(sb-.5,color="gray",ls=":",lw=1.5,label="S1→S2"); a.legend()
    a.set_xlabel("Round"); a.set_ylabel(title)
    a.set_title(f"Base Paper FL — {title}"); a.grid(alpha=.3); a.set_xticks(rounds)
plt.suptitle(f"FL Base Paper Format — {MODEL_NAME}"); plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fl_training_curves.png",dpi=300,bbox_inches="tight"); plt.show()

## 12. Evaluate — 3 Test Conditions

In [ ]:
print("Predicting in-distribution...")
yp_ind = gm.predict(test_ind_ds, verbose=1); yd_ind = np.argmax(yp_ind,1)

print("\nPredicting APTOS clean (unseen)...")
yp_uns = gm.predict(test_uns_ds, verbose=1); yd_uns = np.argmax(yp_uns,1)

print("\nPredicting APTOS+JPEG (unseen degraded)...")
yp_deg = gm.predict(test_deg_ds, verbose=1); yd_deg = np.argmax(yp_deg,1)

## 13. Metrics

In [ ]:
def metrics(yt,yp,yprob,label=""):
    a = accuracy_score(yt,yp)
    p,r,f,_ = precision_recall_fscore_support(yt,yp,average="macro",zero_division=0)
    yb = label_binarize(yt,classes=np.arange(NUM_CLASSES))
    try: au = roc_auc_score(yb,yprob,multi_class="ovr",average="macro")
    except: au = float("nan")
    q = cohen_kappa_score(yt,yp,weights="quadratic")
    return dict(label=label,accuracy=float(a),precision_macro=float(p),
                recall_macro=float(r),f1_macro=float(f),auc_roc=float(au),qwk=float(q))

MKEYS = ["accuracy","precision_macro","recall_macro","f1_macro","auc_roc","qwk"]

m_ind = metrics(y_test_ind, yd_ind, yp_ind, "In-Distribution (DDR+EyePACS)")
m_uns = metrics(y_test_uns, yd_uns, yp_uns, "Unseen Clean (APTOS)")
m_deg = metrics(y_test_uns, yd_deg, yp_deg, "Unseen Degraded (APTOS+JPEG)")

print(f"{'Metric':<18} {'In-Dist':>10} {'Unseen':>10} {'Unseen+JPEG':>12}")
print("-"*52)
for k in MKEYS:
    print(f"{k:<18} {m_ind[k]:>10.4f} {m_uns[k]:>10.4f} {m_deg[k]:>12.4f}")

print("\nGeneralizability Drop (Unseen clean - degraded):")
print("-"*52)
for k in MKEYS:
    d = m_uns[k]-m_deg[k]
    print(f"  {k:<18}: {d:+.4f}")

## 14. Confusion Matrices

In [ ]:
SN=["NoDR","Mild","Mod.","Sev.","Prol."]
cms=[
    (confusion_matrix(y_test_ind,yd_ind,labels=np.arange(NUM_CLASSES)),"In-Distribution"),
    (confusion_matrix(y_test_uns,yd_uns,labels=np.arange(NUM_CLASSES)),"Unseen Clean"),
    (confusion_matrix(y_test_uns,yd_deg,labels=np.arange(NUM_CLASSES)),"Unseen+JPEG"),
]
fig,axes=plt.subplots(1,3,figsize=(18,6))
for ax,(cm,title) in zip(axes,cms):
    im=ax.imshow(cm,cmap="Blues"); plt.colorbar(im,ax=ax,fraction=.046)
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(SN,rotation=30,ha="right"); ax.set_yticklabels(SN)
    th=cm.max()/2
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j,i,str(cm[i,j]),ha="center",va="center",fontsize=8,
                    color="white" if cm[i,j]>th else "black")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
plt.suptitle("FL Base Paper — Confusion Matrices",fontweight="bold")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/confusion_matrices.png",dpi=300,bbox_inches="tight"); plt.show()

## 15. Save Results

In [ ]:
res={
    "experiment":EXPERIMENT_NAME,"model":MODEL_NAME,
    "paradigm":"FedAvg","format":"base_paper",
    "train_clients":TRAIN_CLIENTS,"unseen":UNSEEN_CLIENT,
    "client_sizes":dict(zip(TRAIN_CLIENTS,[int(s) for s in client_sz])),
    "n_test_indist":int(len(y_test_ind)),
    "n_test_unseen":int(len(y_test_uns)),
    "metrics_indist":m_ind,
    "metrics_unseen_clean":m_uns,
    "metrics_unseen_degraded":m_deg,
    "generalizability_drop":{k:float(m_uns[k]-m_deg[k]) for k in MKEYS},
    "config":dict(seed=SEED,lr=LR,momentum=MOMENTUM,batch=BATCH_SIZE,
                  fl_rounds=FL_ROUNDS,local_epochs=LOCAL_EPOCHS,
                  stage1_rounds=STAGE1_ROUNDS,
                  jpeg_quality_min=JPEG_QUALITY_MIN,
                  jpeg_quality_max=JPEG_QUALITY_MAX)
}
with open(f"{LOG_DIR}/results.json","w") as f: json.dump(res,f,indent=2)

row={"experiment":EXPERIMENT_NAME,"paradigm":"FedAvg_BasePaper","model":MODEL_NAME}
row.update({f"indist_{k}":m_ind[k] for k in MKEYS})
row.update({f"unseen_{k}":m_uns[k] for k in MKEYS})
row.update({f"unseen_deg_{k}":m_deg[k] for k in MKEYS})
pd.DataFrame([row]).to_csv(f"{LOG_DIR}/summary_row.csv",index=False)
print(f"Saved to: {RES}")

print("\n"+"="*65)
print(f"  FL BASE PAPER FORMAT — DONE")
print("="*65)
print(f"  Clients: {TRAIN_CLIENTS}")
print(f"  In-Dist  | Acc:{m_ind['accuracy']:.4f}  F1:{m_ind['f1_macro']:.4f}  QWK:{m_ind['qwk']:.4f}")
print(f"  Unseen   | Acc:{m_uns['accuracy']:.4f}  F1:{m_uns['f1_macro']:.4f}  QWK:{m_uns['qwk']:.4f}")
print(f"  +JPEG    | Acc:{m_deg['accuracy']:.4f}  F1:{m_deg['f1_macro']:.4f}  QWK:{m_deg['qwk']:.4f}")
print(f"  Drop     | Acc:{m_uns['accuracy']-m_deg['accuracy']:+.4f}  F1:{m_uns['f1_macro']-m_deg['f1_macro']:+.4f}")
print("="*65)